# TNG100 -> FIREbox domain shift -- seeded 20-run study

20 **recorded** training seeds per method. Graph construction and the train/val/test
split are done **once** and shared by every run, so the only thing varying across runs is
training stochasticity -- not the data.

Two fixes are already applied in this repo (verified present):

1. **Train-test-split fix** (`scripts/process_data.py`) -- `sample_pyg_subgraph` inherits
   each sampled node's original mass-stratified split membership instead of re-assigning a
   fresh unstratified split.
2. **Softmax / potentials fix** (`ShiftKit/shiftkit/methods/sidda.py`) -- `geomloss`
   returns Sinkhorn potentials shaped `(1, N)`; `.softmax(dim=0)` on that shape was a
   no-op (all weights = 1), so `OT-reweight`'s EMA branch was unreachable and
   `Loss-reweight`'s weighting had zero gradient effect. Now `.reshape(-1)` +
   `.softmax(dim=-1)`, with the `ema_ready` check on `shape[-1]`.

Config matches `domain_shift_tng100_4methods.ipynb`: **200 epochs**,
**`potential_temperature=0.5`**, full TNG graph, `tng_to_fire` only.

## Reproducibility

CUDA's scatter-add is non-deterministic by default, so recording a seed is not by itself
enough to reproduce a run. This notebook sets `CUBLAS_WORKSPACE_CONFIG` **before**
importing torch and enables `torch.use_deterministic_algorithms`, which makes SAGEConv
forward/backward bitwise reproducible -- so re-running seed *k* genuinely returns run *k*.
Cell **B2** asserts this rather than assuming it.

## Running order

Training and plotting are decoupled. To **redraw figures without retraining**, run only:

> **B** (setup) -> **H** (load from disk) -> **J** (pick seed) -> **L** (figure)

Cells **D** (graph build) and **F** (training) are only needed to *produce* the artifacts.
Every plot reads from `output/domain-shift-tng100-20seeds/`; nothing downstream of **F**
depends on in-memory training state.


In [ ]:
# NOTE: CUBLAS_WORKSPACE_CONFIG must be set BEFORE torch initialises CUDA,
# otherwise torch.use_deterministic_algorithms() raises on some cuBLAS ops.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm, t as student_t

torch.use_deterministic_algorithms(True, warn_only=True)

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / "scripts").exists() and (candidate / "data").exists():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate repository root containing 'scripts' and 'data'.")

SCRIPTS_DIR = REPO_ROOT / "scripts"
DM_ROOT = REPO_ROOT / "data"
SHIFTKIT_DIR = REPO_ROOT / "ShiftKit"

for path in [str(REPO_ROOT), str(SCRIPTS_DIR), str(DM_ROOT), str(SHIFTKIT_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from process_data import DataProcessor, sample_pyg_subgraph
from process_tng_data import TNGDataProcessor

OUTPUT_DIR = REPO_ROOT / "output" / "domain-shift-tng100-20seeds"
CKPT_DIR = OUTPUT_DIR / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTION = "tng_to_fire"
SRC_NAME, TGT_NAME = "TNG100", "FIREbox"

# 20 recorded seeds. Explicit list (not a count) so it is unambiguous what was run.
SEEDS = list(range(20))

# Training config -- matches domain_shift_tng100_4methods.ipynb
MODEL_CONFIG = {"hidden_channels": 64, "num_layers": 2, "model_name": "SAGE"}
EPOCHS = 200
WARMUP = 10
LR = 5e-3
POTENTIAL_TEMPERATURE = 0.5
OT_EMA_MOMENTUM = 0.9

METHOD_SPECS = [
    dict(label="No-DA",                 kind="sourceonly"),
    dict(label="SIDDA",                 kind="sidda", use_potentials=False, weight_ot=False),
    dict(label="SIDDA + OT-reweight",   kind="sidda", use_potentials=False, weight_ot=True),
    dict(label="SIDDA + Loss-reweight", kind="sidda", use_potentials=True,  weight_ot=False),
]
ALL_METHOD_ORDER = [m["label"] for m in METHOD_SPECS]

METHOD_COLORS = {
    "No-DA": "#4C72B0",
    "SIDDA": "#DD8452",
    "SIDDA + OT-reweight": "#55A868",
    "SIDDA + Loss-reweight": "#C44E52",
}

# Data split / graph construction -- fixed, shared by every run.
CONFIG = {
    "fire_path": DM_ROOT / "firebox_data" / "FIREbox_z=0.txt",
    "tng_path": DM_ROOT / "tng-data" / "TNG100" / "subhalos_99.parquet",
    "r": 1,
    "test_size": 0.1,
    "val_size": 0.1,
    "standardize": True,
    "stratify_bins": 10,
    "random_state": 42,
    "upper_mass": 12,
}
GRAPH_KWARGS = dict(
    r=CONFIG["r"], test_size=CONFIG["test_size"], val_size=CONFIG["val_size"],
    standardize=CONFIG["standardize"], stratify_bins=CONFIG["stratify_bins"],
    random_state=CONFIG["random_state"], upper_mass=CONFIG["upper_mass"],
)

METRICS = ["src_rmse", "src_r2", "src_chi2", "tgt_rmse", "tgt_r2", "tgt_chi2"]
DDOF = 1  # sample std everywhere -- see the stats note below


def npz_path(label, seed):
    return CKPT_DIR / f"{label.replace(' ', '_')}_seed{seed}_predictions.npz"


def compute_metrics(true, mean, std):
    """RMSE, R^2 and reduced chi^2 from a single run's predictions."""
    resid = mean - true
    mse = float(np.mean(resid ** 2))
    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((true - true.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    pulls = resid / std
    return {"rmse": mse ** 0.5, "r2": r2, "chi2": float(np.mean(pulls ** 2))}


print(f"Repo:            {REPO_ROOT}")
print(f"Output:          {OUTPUT_DIR}")
print(f"Seeds:           {SEEDS}")
print(f"Config:          EPOCHS={EPOCHS}  temp={POTENTIAL_TEMPERATURE}  LR={LR}  warmup={WARMUP}")
print(f"CUDA available:  {torch.cuda.is_available()}"
      + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Deterministic:   {torch.are_deterministic_algorithms_enabled()}"
      f"  (CUBLAS_WORKSPACE_CONFIG={os.environ.get('CUBLAS_WORKSPACE_CONFIG')})")


## Build the graphs -- once, shared by all 80 runs

`CONFIG["random_state"] = 42` fixes graph construction and the mass-stratified
train/val/test split. The per-run seeds set later touch **only** training stochasticity,
so every run sees byte-identical data.


In [ ]:
USE_FULL_TNG_GRAPH = True  # False reproduces the size-matched (1,035-node) TNG graph

fire_proc = DataProcessor(file_path=str(CONFIG["fire_path"]), subhalos="both")
fire_data = fire_proc.create_graph_data(**GRAPH_KWARGS)

tng_proc = TNGDataProcessor(file_path=str(CONFIG["tng_path"]), subhalos="both")

MAX_TNG_ROWS_FOR_GRAPH = 7500
if len(tng_proc.df_filtered) > MAX_TNG_ROWS_FOR_GRAPH:
    rng_pre = np.random.default_rng(CONFIG["random_state"])
    keep_idx = np.sort(rng_pre.choice(len(tng_proc.df_filtered), size=MAX_TNG_ROWS_FOR_GRAPH, replace=False))
    print(f"Pre-subsampling TNG100 catalog: {len(tng_proc.df_filtered):,} -> {MAX_TNG_ROWS_FOR_GRAPH:,} rows")
    tng_proc.df_filtered = tng_proc.df_filtered.iloc[keep_idx].reset_index(drop=True)

tng_data = tng_proc.create_graph_data(**GRAPH_KWARGS)

FIRE_MSTAR_COL = "lg_Mstar_<Rhalo"
TNG_MSTAR_COL = "stellar_mass"
fdf = fire_proc.df_filtered
tdf_full = tng_proc.df_filtered
overlap_lo = tdf_full[TNG_MSTAR_COL].values.astype(float).min()
overlap_hi = fdf[FIRE_MSTAR_COL].values.astype(float).max()
print(f"Overlap range: [{overlap_lo:.2f}, {overlap_hi:.2f}] dex in lg_Mstar")

from torch_geometric.data import Data


def induced_subgraph_by_mask(data, keep_mask):
    '''Induced subgraph over nodes where keep_mask is True (split masks carried over).'''
    keep_idx = keep_mask.nonzero(as_tuple=False).view(-1)
    index_map = {int(old): new for new, old in enumerate(keep_idx.tolist())}
    keep_set = set(keep_idx.tolist())

    edge_index = data.edge_index
    src, dst = edge_index[0].tolist(), edge_index[1].tolist()
    edge_mask = torch.tensor([s in keep_set and d in keep_set for s, d in zip(src, dst)], dtype=torch.bool)
    kept_edges = edge_index[:, edge_mask]
    remapped = torch.stack([
        torch.tensor([index_map[int(s)] for s in kept_edges[0].tolist()], dtype=torch.long),
        torch.tensor([index_map[int(d)] for d in kept_edges[1].tolist()], dtype=torch.long),
    ])

    sub = Data(
        x=data.x[keep_idx],
        edge_index=remapped,
        y=data.y[keep_idx],
        pos=data.pos[keep_idx] if getattr(data, "pos", None) is not None else None,
    )
    for split in ("train_mask", "val_mask", "test_mask"):
        if hasattr(data, split):
            setattr(sub, split, getattr(data, split)[keep_idx])
    return sub


fire_keep = torch.tensor((fdf[FIRE_MSTAR_COL].values >= overlap_lo) & (fdf[FIRE_MSTAR_COL].values <= overlap_hi))
tng_keep = torch.tensor((tdf_full[TNG_MSTAR_COL].values >= overlap_lo) & (tdf_full[TNG_MSTAR_COL].values <= overlap_hi))

fire_overlap_graph = induced_subgraph_by_mask(fire_data, fire_keep)
tng_overlap_full_graph = induced_subgraph_by_mask(tng_data, tng_keep)
tng_overlap_matched_graph = sample_pyg_subgraph(
    tng_overlap_full_graph, num_nodes_sample=fire_overlap_graph.num_nodes,
    random_state=CONFIG["random_state"],
)
tng_graph = tng_overlap_full_graph if USE_FULL_TNG_GRAPH else tng_overlap_matched_graph
tng_graph_label = "TNG100 (overlap, full)" if USE_FULL_TNG_GRAPH else "TNG100 (overlap, size-matched)"

for name, d in [("FIREbox (overlap)", fire_overlap_graph), (tng_graph_label, tng_graph)]:
    print(f"{name:32s} nodes={d.num_nodes:,}  edges={d.num_edges:,}  "
          f"train={int(d.train_mask.sum())}  val={int(d.val_mask.sum())}  test={int(d.test_mask.sum())}")

# The split must be a genuine partition, and must come from DataProcessor's stratified
# split rather than any fallback -- fail loudly if not.
for name, d in [("FIREbox", fire_overlap_graph), (tng_graph_label, tng_graph)]:
    masks = [d.train_mask, d.val_mask, d.test_mask]
    assert all(m is not None and bool(m.any()) for m in masks), f"{name}: missing/empty split mask"
    assert sum(int(m.sum()) for m in masks) == d.num_nodes, f"{name}: masks do not partition the nodes"
    assert not bool((d.train_mask & d.val_mask).any() or (d.train_mask & d.test_mask).any()
                    or (d.val_mask & d.test_mask).any()), f"{name}: split masks overlap"
    frac = int(d.test_mask.sum()) / d.num_nodes
    assert abs(frac - CONFIG["test_size"]) < 0.03, (
        f"{name}: test fraction {frac:.3f} != CONFIG['test_size'] {CONFIG['test_size']}")
print("split masks verified: stratified partition, shared by every run")

DIRECTIONS = {
    DIRECTION: dict(src=tng_graph, tgt=fire_overlap_graph, src_name=SRC_NAME, tgt_name=TGT_NAME),
}


## B2 -- verify determinism actually holds

Recorded seeds are only meaningful if the same seed reproduces the same run. Train the
cheapest method twice at the same seed and require the predictions to be **bitwise**
identical. If this fails, the seeds are labels rather than reproducibility handles, and
that needs to be known before spending hours on the sweep.


In [ ]:
from shiftkit import GNN, DataManager, SourceOnlyGaussianRegressionTrainer

_spec = DIRECTIONS[DIRECTION]
_dm = DataManager(batch_size=1, num_workers=0)
_train_src, _train_tgt = _dm.load("pyg_domains", train=True, source=_spec["src"], target=_spec["tgt"])
_test_src, _test_tgt = _dm.load("pyg_domains", train=False, source=_spec["src"], target=_spec["tgt"])


def _one_short_run(seed, epochs=12):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = GNN(_spec["src"], MODEL_CONFIG["model_name"],
                hidden_channels=MODEL_CONFIG["hidden_channels"],
                num_layers=MODEL_CONFIG["num_layers"],
                regress=True, pool="none", predict_var=True)
    tr = SourceOnlyGaussianRegressionTrainer(model, _train_src, _train_tgt, lr=LR)
    tr.fit(epochs=epochs)
    _, mean_tgt, std_tgt = tr.predict(_test_tgt)
    return mean_tgt, std_tgt


_m1, _s1 = _one_short_run(1234)
_m2, _s2 = _one_short_run(1234)
_same_mean = np.array_equal(_m1, _m2)
_same_std = np.array_equal(_s1, _s2)
print(f"same seed twice -> mean bitwise identical: {_same_mean}")
print(f"same seed twice -> std  bitwise identical: {_same_std}")
print(f"max |mean diff| = {np.abs(_m1 - _m2).max():.3e}")

_m3, _ = _one_short_run(5678)
print(f"different seed  -> mean differs (sanity): {not np.array_equal(_m1, _m3)}"
      f"   max |diff| = {np.abs(_m1 - _m3).max():.3e}")

assert _same_mean and _same_std, (
    "DETERMINISM CHECK FAILED -- the same seed did not reproduce the same run. "
    "Recorded seeds would be labels only. Investigate before running the full sweep."
)
print("\nOK: seeds are genuine reproducibility handles.")


## Train: 4 methods x 20 seeds

Each run is reseeded immediately before model construction, so run *(method, seed)* is a
deterministic function of `seed` alone -- independent of what earlier runs consumed from
the global RNG. Predictions are written to disk per run, and a CSV row is appended after
each run so an interrupted sweep keeps its completed work.


In [ ]:
import time
import csv

from shiftkit import (
    GNN, DataManager,
    SourceOnlyGaussianRegressionTrainer, SIDDAGaussianRegressionTrainer,
)

progress_log = OUTPUT_DIR / "progress.log"
partial_csv = OUTPUT_DIR / "per_seed_results_partial.csv"


def log_progress(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(progress_log, "a") as f:
        f.write(line + "\n")


def build_trainer(spec, model, train_src, train_tgt):
    if spec["kind"] == "sourceonly":
        return SourceOnlyGaussianRegressionTrainer(model, train_src, train_tgt, lr=LR)
    if spec["kind"] == "sidda":
        return SIDDAGaussianRegressionTrainer(
            model, train_src, train_tgt, lr=LR, warmup_epochs=WARMUP,
            use_potentials=spec["use_potentials"], weight_ot=spec["weight_ot"],
            potential_temperature=POTENTIAL_TEMPERATURE, ot_ema_momentum=OT_EMA_MOMENTUM,
        )
    raise ValueError(spec["kind"])


spec = DIRECTIONS[DIRECTION]
src, tgt = spec["src"], spec["tgt"]
dm = DataManager(batch_size=1, num_workers=0)
train_src, train_tgt = dm.load("pyg_domains", train=True, source=src, target=tgt)
test_src, test_tgt = dm.load("pyg_domains", train=False, source=src, target=tgt)

if not partial_csv.exists():
    with open(partial_csv, "w", newline="") as f:
        csv.writer(f).writerow(["method", "seed"] + METRICS + ["elapsed_s"])

t_sweep = time.time()
for method_spec in METHOD_SPECS:
    label = method_spec["label"]
    for seed in SEEDS:
        t0 = time.time()

        # Reseed immediately before construction: makes (method, seed) a deterministic
        # function of `seed` alone, not of the accumulated RNG state.
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        model = GNN(
            src, MODEL_CONFIG["model_name"],
            hidden_channels=MODEL_CONFIG["hidden_channels"],
            num_layers=MODEL_CONFIG["num_layers"],
            regress=True, pool="none", predict_var=True,
        )
        trainer = build_trainer(method_spec, model, train_src, train_tgt)
        trainer.fit(epochs=EPOCHS)
        elapsed = time.time() - t0

        true_src, mean_src, std_src = trainer.predict(test_src)
        true_tgt, mean_tgt, std_tgt = trainer.predict(test_tgt)

        np.savez(
            npz_path(label, seed),
            true_src=true_src, mean_src=mean_src, std_src=std_src,
            true_tgt=true_tgt, mean_tgt=mean_tgt, std_tgt=std_tgt,
            seed=np.array([seed]),
        )

        ms = compute_metrics(true_src, mean_src, std_src)
        mt = compute_metrics(true_tgt, mean_tgt, std_tgt)
        with open(partial_csv, "a", newline="") as f:
            csv.writer(f).writerow([
                label, seed, ms["rmse"], ms["r2"], ms["chi2"],
                mt["rmse"], mt["r2"], mt["chi2"], elapsed,
            ])

        log_progress(
            f"{label} | seed {seed:>2} : {elapsed:6.1f}s  "
            f"src R2={ms['r2']:.3f} chi2={ms['chi2']:.3f}  |  "
            f"tgt R2={mt['r2']:.3f} chi2={mt['chi2']:.3f}"
        )

log_progress(f"=== sweep complete in {(time.time()-t_sweep)/60:.1f} min ===")


## Load results from disk and summarise

Metrics are **recomputed from the saved `.npz` predictions**, which are the source of
truth; the CSVs are derived. That way the tables and the figures can never disagree, and
this cell runs standalone on a fresh kernel without retraining.

### Error bars

Three different quantities get drawn as error bars in this work; they are not
interchangeable:

| Quantity | Meaning |
|---|---|
| `sigma_pred` (per-node, scatter panels) | the model's own predicted std. **+/-1 sigma, not a 68% CI** -- the moment-network loss is likelihood-free, and chi^2 > 1 means it is optimistic. The pull panels are the calibration check. |
| across-seed **SD** | how much a single training run varies. `ddof=1`. |
| across-seed **95% CI** | how well the *mean* is known: `+/- t(0.975, 19) * SD/sqrt(20)` = `+/-2.093 * SEM`. |

At n=20 the SD and the CI half-width differ by ~4.5x, so both are reported.
`summary_stats.csv` stores mean / std / sem / ci95_lo / ci95_hi / median for every metric,
so the paper can switch convention later without re-running anything.


In [ ]:
rows = []
missing = []
for label in ALL_METHOD_ORDER:
    for seed in SEEDS:
        p = npz_path(label, seed)
        if not p.exists():
            missing.append((label, seed))
            continue
        z = np.load(p)
        ms = compute_metrics(z["true_src"], z["mean_src"], z["std_src"])
        mt = compute_metrics(z["true_tgt"], z["mean_tgt"], z["std_tgt"])
        rows.append({
            "method": label, "seed": seed,
            "src_rmse": ms["rmse"], "src_r2": ms["r2"], "src_chi2": ms["chi2"],
            "tgt_rmse": mt["rmse"], "tgt_r2": mt["r2"], "tgt_chi2": mt["chi2"],
        })

if missing:
    print(f"WARNING: {len(missing)} missing runs: {missing[:10]}{' ...' if len(missing) > 10 else ''}")

per_seed_df = pd.DataFrame(rows).sort_values(["method", "seed"]).reset_index(drop=True)

# Completeness / integrity checks
n_expected = len(ALL_METHOD_ORDER) * len(SEEDS)
assert len(per_seed_df) == n_expected, f"expected {n_expected} runs, found {len(per_seed_df)}"
for label in ALL_METHOD_ORDER:
    seen = sorted(per_seed_df[per_seed_df.method == label]["seed"].tolist())
    assert seen == sorted(SEEDS), f"{label}: seed set mismatch -> {seen}"

# Every run must have trained/tested on identical data.
_ref = np.load(npz_path(ALL_METHOD_ORDER[0], SEEDS[0]))
for label in ALL_METHOD_ORDER:
    for seed in SEEDS:
        z = np.load(npz_path(label, seed))
        assert np.array_equal(z["true_tgt"], _ref["true_tgt"]), f"{label}/{seed}: target labels differ"
        assert np.array_equal(z["true_src"], _ref["true_src"]), f"{label}/{seed}: source labels differ"
print(f"verified: {n_expected} runs, {len(SEEDS)} unique seeds/method, identical data across all runs")
print(f"          n_test source={_ref['true_src'].size}  target={_ref['true_tgt'].size}")

per_seed_path = OUTPUT_DIR / "per_seed_results.csv"
per_seed_df.to_csv(per_seed_path, index=False)

# ── summary: mean / std(ddof=1) / sem / 95% CI / median ────────────────────
n = len(SEEDS)
tcrit = float(student_t.ppf(0.975, df=n - 1))
summary_rows = []
for label in ALL_METHOD_ORDER:
    sub = per_seed_df[per_seed_df.method == label]
    row = {"method": label, "n": n}
    for m in METRICS:
        v = sub[m].to_numpy()
        mean, sd = float(v.mean()), float(v.std(ddof=DDOF))
        sem = sd / np.sqrt(n)
        row[f"{m}_mean"] = mean
        row[f"{m}_std"] = sd
        row[f"{m}_sem"] = sem
        row[f"{m}_ci95_lo"] = mean - tcrit * sem
        row[f"{m}_ci95_hi"] = mean + tcrit * sem
        row[f"{m}_median"] = float(np.median(v))
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).set_index("method")
summary_df.to_csv(OUTPUT_DIR / "summary_stats.csv")

print(f"\nmean +/- SD [95% CI],  n={n},  t(0.975,{n-1})={tcrit:.3f}\n" + "=" * 92)
for label in ALL_METHOD_ORDER:
    r = summary_df.loc[label]
    print(f"\n{label}")
    for dom in ("src", "tgt"):
        print(f"  {dom}:  "
              f"RMSE {r[f'{dom}_rmse_mean']:.4f} +/- {r[f'{dom}_rmse_std']:.4f} "
              f"[{r[f'{dom}_rmse_ci95_lo']:.4f}, {r[f'{dom}_rmse_ci95_hi']:.4f}]   "
              f"R2 {r[f'{dom}_r2_mean']:+.3f} +/- {r[f'{dom}_r2_std']:.3f} "
              f"[{r[f'{dom}_r2_ci95_lo']:+.3f}, {r[f'{dom}_r2_ci95_hi']:+.3f}]")

# chi^2 is heavy-tailed when the uncertainty head collapses -- a CI on its mean is not
# trustworthy, so report the median and the outlier count alongside it.
CHI2_OUTLIER = 10.0
print("\n" + "=" * 92)
print(f"chi^2 detail (CI unreliable here -- heavy tailed; outlier := chi^2 >= {CHI2_OUTLIER})")
print(f"{'method':<24} {'dom':>4} {'mean':>10} {'median':>9} {'n_outlier':>10} {'mean excl.':>11}")
print("-" * 92)
for label in ALL_METHOD_ORDER:
    sub = per_seed_df[per_seed_df.method == label]
    for dom in ("src", "tgt"):
        v = sub[f"{dom}_chi2"].to_numpy()
        clean = v[v < CHI2_OUTLIER]
        n_out = int((v >= CHI2_OUTLIER).sum())
        excl = f"{clean.mean():.3f}" if clean.size else "n/a"
        print(f"{label:<24} {dom:>4} {v.mean():>10.3f} {np.median(v):>9.3f} "
              f"{n_out:>10} {excl:>11}")
        if n_out:
            bad = sorted(sub.loc[v >= CHI2_OUTLIER, "seed"].tolist())
            print(f"{'':<24} {'':>4} -> outlier seeds: {bad}")

print(f"\nsaved {per_seed_path}")
print(f"saved {OUTPUT_DIR / 'summary_stats.csv'}")


## Pick a representative seed

One seed is shown in the figure, so the figure reads as a single real training run rather
than a composite. The seed is chosen by distance to each method's **mean target R^2**,
normalised by that method's own SD so "closeness" is comparable across methods with
different spreads, then weighted -- `No-DA` is down-weighted so the SIDDA variants drive
the choice.

Per-method signed deviations at the winning seed are printed so the choice can be checked
rather than trusted. To override, set `REPRESENTATIVE_SEED_OVERRIDE` and re-run this cell
plus the figure cell -- no retraining.


In [ ]:
SEED_PICK_WEIGHTS = {
    "No-DA": 0.25,
    "SIDDA": 1.0,
    "SIDDA + OT-reweight": 1.0,
    "SIDDA + Loss-reweight": 1.0,
}
REPRESENTATIVE_SEED_OVERRIDE = 2  # set to None to auto-pick the most typical seed

stats = {
    label: (
        per_seed_df[per_seed_df.method == label]["tgt_r2"].mean(),
        per_seed_df[per_seed_df.method == label]["tgt_r2"].std(ddof=DDOF),
    )
    for label in ALL_METHOD_ORDER
}

score_rows = []
for seed in SEEDS:
    num = den = 0.0
    per_method = {}
    for label in ALL_METHOD_ORDER:
        v = per_seed_df[(per_seed_df.method == label) & (per_seed_df.seed == seed)]["tgt_r2"].iloc[0]
        mean, sd = stats[label]
        z = (v - mean) / sd if sd > 0 else 0.0
        per_method[label] = z
        w = SEED_PICK_WEIGHTS[label]
        num += w * abs(z)
        den += w
    score_rows.append({"seed": seed, "score": num / den,
                       **{f"z::{k}": v for k, v in per_method.items()}})

score_df = pd.DataFrame(score_rows).sort_values("score").reset_index(drop=True)
score_df.to_csv(OUTPUT_DIR / "seed_selection_scores.csv", index=False)

print("Weighted distance from each method's mean target R^2 (in SDs; lower = more typical)")
print(f"weights: {SEED_PICK_WEIGHTS}\n")
hdr = f"{'seed':>5} {'score':>8}  " + "  ".join(f"{l[:18]:>18}" for l in ALL_METHOD_ORDER)
print(hdr); print("-" * len(hdr))
for _, r in score_df.iterrows():
    print(f"{int(r['seed']):>5} {r['score']:>8.3f}  "
          + "  ".join(f"{r[f'z::{l}']:>+18.2f}" for l in ALL_METHOD_ORDER))

REPRESENTATIVE_SEED = (int(score_df.iloc[0]["seed"]) if REPRESENTATIVE_SEED_OVERRIDE is None
                       else int(REPRESENTATIVE_SEED_OVERRIDE))
_src = "auto (most typical)" if REPRESENTATIVE_SEED_OVERRIDE is None else "manual override"
print(f"\nREPRESENTATIVE_SEED = {REPRESENTATIVE_SEED}   [{_src}]")
print("\nAt this seed, per method (target R^2):")
print(f"{'method':<24} {'this seed':>10} {'mean':>10} {'SD':>8} {'deviation':>12}")
print("-" * 68)
for label in ALL_METHOD_ORDER:
    v = per_seed_df[(per_seed_df.method == label)
                    & (per_seed_df.seed == REPRESENTATIVE_SEED)]["tgt_r2"].iloc[0]
    mean, sd = stats[label]
    print(f"{label:<24} {v:>+10.3f} {mean:>+10.3f} {sd:>8.3f} {(v-mean)/sd:>+10.2f} SD")


## Combined figure: scatter (rows 1-2) + pulls (rows 3-4)

One representative seed. Method name appears once, on the top row; every metric is drawn
inside its panel to keep the vertical space for data.

Scatter error bars are **+/-1 predicted sigma** (not a 68% interval -- see the note above);
the pull row directly beneath each scatter row is the calibration check for those bars.


In [ ]:
from analysis.plot_style import plot_pred_vs_true_with_uncertainty

# plot_style sets font.size=11 on import; override AFTER importing it.
plt.rcParams.update({"font.size": 18})

seed = REPRESENTATIVE_SEED
FIT_COLOR = "#E69F00"
bin_edges = np.linspace(-4, 4, 41)
x_grid = np.linspace(-4, 4, 400)

# ── gather this seed's predictions ────────────────────────────────────────
panel = {}
for label in ALL_METHOD_ORDER:
    z = np.load(npz_path(label, seed))
    panel[label] = {
        "tgt": (z["true_tgt"], z["mean_tgt"], z["std_tgt"]),
        "src": (z["true_src"], z["mean_src"], z["std_src"]),
    }

# Common scatter limits across every scatter panel, so columns are comparable.
_all = np.concatenate([np.concatenate([panel[l][d][0], panel[l][d][1]])
                       for l in ALL_METHOD_ORDER for d in ("tgt", "src")])
s_lo, s_hi = float(_all.min()) - 0.25, float(_all.max()) + 0.25

ncol = len(ALL_METHOD_ORDER)
fig = plt.figure(figsize=(20, 16))

# Nested gridspec so the two rows WITHIN a block sit close together (they share an
# x-axis, and the upper row's tick labels are hidden), while the scatter block and
# the pull block stay visually separated. A single uniform grid cannot do both.
outer = fig.add_gridspec(2, 1, hspace=0.13, left=0.105, right=0.995, top=0.935, bottom=0.055)
gs_scatter = outer[0].subgridspec(2, ncol, hspace=0.035, wspace=0.06)
gs_pulls = outer[1].subgridspec(2, ncol, hspace=0.035, wspace=0.06)

axes = np.empty((4, ncol), dtype=object)
for r in range(2):
    for c in range(ncol):
        axes[r, c] = fig.add_subplot(gs_scatter[r, c])
        axes[2 + r, c] = fig.add_subplot(gs_pulls[r, c])

for col, label in enumerate(ALL_METHOD_ORDER):
    color = METHOD_COLORS[label]

    for block, dom in enumerate(("tgt", "src")):            # rows 0,1 = scatter
        true, mean, std = panel[label][dom]
        m = compute_metrics(true, mean, std)
        ax = axes[block, col]
        plot_pred_vs_true_with_uncertainty(
            true, mean, std, ax, color=color, markersize=3.5, annotate_fontsize=17,
            annotate=f"R$^2$={m['r2']:.3f}\nRMSE={m['rmse']:.3f}",
        )
        ax.set_xlim(s_lo, s_hi); ax.set_ylim(s_lo, s_hi)
        ax.grid(False)

    for block, dom in enumerate(("tgt", "src")):            # rows 2,3 = pulls
        true, mean, std = panel[label][dom]
        pulls = (mean - true) / std
        chi2 = float(np.mean(pulls ** 2))
        mu, sigma = norm.fit(pulls)
        ax = axes[2 + block, col]
        ax.hist(pulls, bins=bin_edges, density=True, alpha=0.85, color=color)
        ax.plot(x_grid, norm.pdf(x_grid, mu, sigma), color=FIT_COLOR, lw=1.8,
                label=f"Fit: $\\mu$={mu:.2f}, $\\sigma$={sigma:.2f}")
        ax.plot(x_grid, norm.pdf(x_grid), color="k", lw=1.3, ls="--",
                label=r"$\mathcal{N}(0,1)$")
        # legend kept a little below the 18 pt body text so the fit line fits the panel
        ax.legend(fontsize=14, frameon=False, loc="upper left", handlelength=1.0,
                  borderpad=0.12, labelspacing=0.22, borderaxespad=0.3)
        ax.text(0.97, 0.97, f"$\\chi^2$={chi2:.2f}", transform=ax.transAxes,
                va="top", ha="right", fontsize=17)
        ax.set_xlim(-4, 4); ax.set_ylim(0, 0.9); ax.grid(False)

    # Method name once, on the top row only.
    axes[0, col].set_title(label, fontsize=19, fontweight="bold", pad=10)

# ── row identity on the left column; units labelled once per 2-row block ──
axes[0, 0].set_ylabel(f"Target ({TGT_NAME})", fontsize=18)
axes[1, 0].set_ylabel(f"Source ({SRC_NAME})", fontsize=18)
axes[2, 0].set_ylabel(f"Target ({TGT_NAME})", fontsize=18)
axes[3, 0].set_ylabel(f"Source ({SRC_NAME})", fontsize=18)

for col in range(ncol):
    axes[1, col].set_xlabel(r"true $\log M_{halo}/M_\odot$", fontsize=18)
    axes[3, col].set_xlabel(r"Pull  $(\hat{\mu} - y)\,/\,\hat{\sigma}$", fontsize=18)
    axes[0, col].tick_params(labelbottom=False)   # shares x with row 1
    axes[2, col].tick_params(labelbottom=False)   # shares x with row 3
    if col > 0:
        for row in range(4):
            axes[row, col].tick_params(labelleft=False)

fig.suptitle(
    f"{SRC_NAME} $\\to$ {TGT_NAME}  --  representative seed {seed} of {len(SEEDS)}   "
    f"(scatter bars = $\\pm1\\,\\hat{{\\sigma}}$ predicted, not a 68% CI)",
    fontweight="bold", fontsize=19,
)

# Block-level y-axis labels: one per pair of rows. Positions read off the gridspec,
# which is already final (no tight_layout to shift things afterwards).
def _block_centre(rows):
    tops = [axes[r, 0].get_position().y1 for r in rows]
    bots = [axes[r, 0].get_position().y0 for r in rows]
    return (max(tops) + min(bots)) / 2.0

_x = min(axes[r, 0].get_position().x0 for r in range(4)) - 0.078
fig.text(_x, _block_centre([0, 1]), r"predicted $\log M_{halo}/M_\odot$",
         rotation="vertical", va="center", ha="center", fontsize=19)
fig.text(_x, _block_centre([2, 3]), "Density",
         rotation="vertical", va="center", ha="center", fontsize=19)

stem = OUTPUT_DIR / f"{DIRECTION}_seed{seed}_scatter_pulls"
fig.savefig(stem.with_suffix(".png"), dpi=150, bbox_inches="tight")
fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
print("saved", stem.with_suffix(".png"))
print("saved", stem.with_suffix(".pdf"))
plt.show()


## Across-seed comparison: SD and 95% CI

Wide light bar = **+/-1 SD** across the 20 seeds (how much a single run varies).
Narrow dark bar = **95% CI on the mean** (how well the mean is known). At n=20 these
differ by ~4.5x, so showing only one would misstate how separated the methods are.


In [ ]:
n = len(SEEDS)
tcrit = float(student_t.ppf(0.975, df=n - 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (metric, ylabel) in zip(
    axes, [("tgt_rmse", "target RMSE"), ("tgt_r2", "target R$^2$"), ("tgt_chi2", "target $\\chi^2$")]
):
    for x, label in enumerate(ALL_METHOD_ORDER):
        v = per_seed_df[per_seed_df.method == label][metric].to_numpy()
        mean, sd = v.mean(), v.std(ddof=DDOF)
        ci = tcrit * sd / np.sqrt(n)
        color = METHOD_COLORS[label]
        jitter = np.random.default_rng(0).normal(0, 0.06, size=v.size)
        ax.scatter(np.full(v.size, x) + jitter, v, s=16, alpha=0.35, color=color, zorder=2)
        ax.errorbar(x, mean, yerr=sd, fmt="none", ecolor=color, elinewidth=6,
                    alpha=0.30, capsize=0, zorder=3)              # +/-1 SD
        ax.errorbar(x, mean, yerr=ci, fmt="o", ms=8, color=color, ecolor=color,
                    elinewidth=2.2, capsize=5, mec="k", mew=0.8, zorder=4)  # 95% CI
    ax.set_xticks(range(len(ALL_METHOD_ORDER)))
    ax.set_xticklabels(ALL_METHOD_ORDER, rotation=20, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25, lw=0.5)
    if metric == "tgt_r2":
        ax.axhline(0, color="k", lw=0.6, ls="--", alpha=0.5)
    if metric == "tgt_chi2":
        ax.axhline(1, color="k", lw=0.6, ls="--", alpha=0.5)
        # chi^2 outliers can be ~1000x the typical value; log scale keeps the panel readable.
        if per_seed_df[metric].max() / max(per_seed_df[metric].median(), 1e-9) > 20:
            ax.set_yscale("log")
            ax.set_ylabel(ylabel + "  (log scale)")

fig.suptitle(
    f"{SRC_NAME} $\\to$ {TGT_NAME}, {n} seeds   "
    "(wide pale bar = $\\pm$1 SD across seeds;  dark bar = 95% CI on the mean)",
    fontsize=11,
)
fig.tight_layout()
cmp_path = OUTPUT_DIR / f"{DIRECTION}_{n}seeds_sd_ci_comparison.png"
fig.savefig(cmp_path, dpi=150, bbox_inches="tight")
fig.savefig(cmp_path.with_suffix(".pdf"), bbox_inches="tight")
print("saved", cmp_path)
plt.show()
